# Noise explorer

Every drone / rotor ego-noise dataset we hold, its recordings with their metadata, any recording loaded by id as a `td.Frame`, and a Plotly spectrogram + spectrum viewer over it.

Logic lives in `src/`: `plots.explore.noise_datasets` / `noise_recordings` / `load_recording` and `plots.spectrum_viewer.spectrum_viewer`. The first listing of a dataset streams each shard once; the per-recording index is cached under `.cache/noise_explorer/`, so later listings and loads are instant (a load streams only the recording's shard).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")  # R2 credentials for dload

import ipywidgets as W
import numpy as np
import pandas as pd
from IPython.display import Audio, display

from plots import explore
from plots.spectrum_viewer import spectrum_viewer
from utils.audio import first_channel

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60)

## 1 · Drone-noise datasets

`kind = frames`: published recording Frames, loadable below. `kind = raw`: file trees without a frames builder, listed for completeness. `pinned` is the `dload.lock` version (`None` = not pinned yet).

In [ ]:
catalog = explore.noise_datasets()
catalog

## 2 · Recordings of one dataset

One row per recording: id, shard, channels, sample rate, duration, entry names, then every `meta` field (nested keys dotted, e.g. `system.make_model`). `MAX_SHARDS` bounds the first scan of the very large sets (DroneAudioSet: 552 shards, 88 GiB; drone-detection-samples: 106 shards); `recs.attrs["shards"]` says how much was indexed.

In [ ]:
DATASET = "DREGON-frames"
MAX_SHARDS = None  # e.g. 2 for DroneAudioSet / drone-detection-samples

recs = explore.noise_recordings(DATASET, max_shards=MAX_SHARDS)
print(
    f"{DATASET}@{recs.attrs['version']}: {len(recs)} recordings, shards indexed {recs.attrs['shards']}"
)
recs

Plain pandas from here, e.g. filter on any meta column:

In [ ]:
print(list(recs.columns))
recs[recs["recording_id"].str.contains("room1")].head(10)

## 3 · Load a recording by id

`load_recording` returns the Frame exactly as published: audio plus every telemetry track, meta nested under `meta`.

In [ ]:
REC_ID = "free-flight_nosource_room1"

frame = explore.load_recording(DATASET, REC_ID)
print([k for k, _ in frame.items()])
explore.meta_table([frame]).T

## 4 · Spectrogram + spectrum

Controls:
- **spectrum**: `overall (FFT)` — one FFT over the whole (cropped) channel; `mean (STFT)` — the RMS over the spectrogram's frames; `frame` — one spectrogram column. **Click the spectrogram** to pin a frame (cyan line), or move the `frame t` slider.
- **freq axis**: linear / log, on both panels. **amplitude**: dB / linear, on both panels (`dyn range` sets the dB colour span).
- **freq (Hz)**: the band shown on both panels; the overall FFT is re-decimated for the band, so narrowing it shows more bins. Plotly zoom/pan also works on either panel.

Amplitudes are normalised so a sinusoid of amplitude A reads A (0 dB = unit amplitude) in every mode. `start_s` / `duration_s` crop long recordings (a Frame's own time axis); `n_fft`, `hop`, `channel` are keywords too.

In [ ]:
viewer = spectrum_viewer(frame, duration_s=30.0, fmax=4000)
display(viewer)

ch0 = first_channel(frame["audio"])
sr = int(ch0.tindex.sr)
display(Audio(np.asarray(ch0.data)[: 30 * sr], rate=sr))

## 5 · Pick with dropdowns

Dataset → recording → viewer, without editing cells. Large sets are indexed on their first `PICKER_MAX_SHARDS` shards only.

In [ ]:
PICKER_MAX_SHARDS = 2  # for the sets marked "index a few shards"
PICKER_CROP_S = 60.0  # the viewer crops to this many seconds from the start
LARGE = {"DroneAudioSet", "drone-detection-samples"}

frames_sets = catalog.loc[
    (catalog["kind"] == "frames") & catalog["pinned"].notna(), "name"
].tolist()
w_set = W.Dropdown(options=frames_sets, description="dataset:", layout=W.Layout(width="360px"))
w_rec = W.Dropdown(description="recording:", layout=W.Layout(width="520px"))
out = W.Output()


def _on_set(_=None):
    table = explore.noise_recordings(
        w_set.value, max_shards=PICKER_MAX_SHARDS if w_set.value in LARGE else None
    )
    w_rec.options = table["recording_id"].tolist()


def _on_rec(_=None):
    if w_rec.value is None:
        return
    with out:
        out.clear_output(wait=True)
        fr = explore.load_recording(w_set.value, w_rec.value)
        display(explore.meta_table([fr]).T)
        display(spectrum_viewer(fr, duration_s=PICKER_CROP_S))


w_set.observe(_on_set, "value")
w_rec.observe(_on_rec, "value")
display(W.HBox([w_set, w_rec]), out)
_on_set()